# 03 · Explain the risk

Which factors raise or lower a loan's risk score, using SHAP on the final model of a
chosen run (trained on every loan with a known outcome). This is the explanation the
officials will see in the M4 dashboard; the decision stays with them.

> The table holds member codes and features only: no names, no free text (decision D-016). The default label is **provisional** until the Chama agrees a definition (D-018): `defaulted` = rolled over while unpaid, overdue, or lost track.

In [ ]:
# Setup: runs in Google Colab and on this computer.
# Colab, once: add a GitHub token that can read this repository as the Colab
# secret GITHUB_TOKEN (key icon on the left). This cell mounts Drive, fetches
# the code and installs what Colab lacks. On this computer it uses DRIVE_DIR
# from .env and the `chama` conda environment.
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "m2-features-models"

if IN_COLAB:
    from google.colab import drive, userdata

    REPO = "/content/chama-loan-default"
    PLAIN_URL = "https://github.com/muriithi-alvin/chama-loan-default.git"

    def git(*args):
        """Run git with the token, without ever printing it."""
        url = PLAIN_URL.replace("https://", f"https://x-access-token:{userdata.get('GITHUB_TOKEN')}@")
        result = subprocess.run(["git", *[url if a == "URL" else a for a in args]],
                                capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError("git failed: check the GITHUB_TOKEN secret and the branch name")

    drive.mount("/content/drive")
    if os.path.exists(REPO):
        git("-C", REPO, "pull", "-q", "URL", BRANCH)
    else:
        git("clone", "-q", "--branch", BRANCH, "URL", REPO)
        git("-C", REPO, "remote", "set-url", "origin", PLAIN_URL)   # no token left behind
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mlflow", "shap", "imbalanced-learn", "python-dotenv"], check=True)
    sys.path.insert(0, f"{REPO}/src")
    os.environ["DRIVE_DIR"] = "/content/drive/MyDrive/chama-loan-default"

import warnings

import pandas as pd

from chama.config import export_dir, load_config, mlflow_dir
from chama.features.export import FILE_NAME

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
CONFIG = load_config()
SETTINGS = CONFIG["modelling"]
DATA = export_dir() / FILE_NAME
print("data:  ", DATA)
print("MLflow:", mlflow_dir())

In [ ]:
from chama.models.train import setup_mlflow

mlflow = setup_mlflow(mlflow_dir(), SETTINGS["experiment"])
runs = mlflow.search_runs(order_by=["metrics.forward_pr_auc DESC"])
runs[["tags.mlflow.runName", "metrics.forward_pr_auc", "metrics.forward_brier",
      "metrics.members_pr_auc", "start_time"]].head(12)

In [ ]:
# The run to explain: the best walk-forward PR AUC, or set RUN_NAME to another one.
RUN_NAME = None
chosen = runs.iloc[0] if RUN_NAME is None else runs[runs["tags.mlflow.runName"] == RUN_NAME].iloc[0]
model = mlflow.sklearn.load_model(f"runs:/{chosen['run_id']}/model")
print("explaining:", chosen["tags.mlflow.runName"])

In [ ]:
from chama.models import plots
from chama.models.data import known_outcomes, load_features, xy
from chama.models.explain import importance, shap_values

df = load_features(DATA)
X_known, _ = xy(known_outcomes(df))
contributions = shap_values(model, X_known, background=X_known)
plots.importance_bar(importance(contributions), chosen["tags.mlflow.runName"])

## Loans still open

Loans whose outcome is not known yet are what the model is for: a score, a risk band
and the three strongest factors for each.

In [ ]:
from IPython.display import display

from chama.models.evaluate import risk_band
from chama.models.explain import top_factors

open_loans = df[~df["in_training"]]
X_open, _ = xy(open_loans)
scored = open_loans[["loan_id", "member_code", "loan_type", "issued_month", "amount"]].assign(
    score=model.predict_proba(X_open)[:, 1].round(3))
scored["band"] = risk_band(scored["score"], SETTINGS["risk_bands"])
scored.sort_values("score", ascending=False)

In [ ]:
open_contributions = shap_values(model, X_open, background=X_known)
for loan_id, row in open_contributions.iterrows():
    print(scored.loc[loan_id, "loan_id"], "| score", scored.loc[loan_id, "score"],
          "|", scored.loc[loan_id, "band"])
    display(top_factors(row))